# EfficientNet-B0

??????? ??? ????????, ?????? ? ????????? ??????????? ?????? `efficientnet_b0` ?? APTOS 2019.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project directory:", PROJECT_DIR)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import torch

from config import DEVICE, FIGURES_DIR, METRICS_DIR, SAVED_MODELS_DIR
from dataset import get_class_weights
from models import get_model
from utils import count_parameters, get_gpu_info

MODEL_NAME = "efficientnet_b0"
FREEZE_BACKBONE = True

print("Device:", DEVICE)
print("GPU info:", get_gpu_info())
print("Class weights:", get_class_weights().tolist())

In [ ]:
model = get_model(MODEL_NAME, freeze_backbone=FREEZE_BACKBONE)
print(model.__class__.__name__)
print("Parameters:", f"{count_parameters(model):,}")
print("Trainable parameters:", f"{count_parameters(model, trainable_only=True):,}")

## ????????

?????? ???? ????????? ???????? training script ?? `src/train.py`. ?????????? ??????????? ? `results/`.

In [ ]:
from train import train

train(model_name=MODEL_NAME, freeze_backbone=FREEZE_BACKBONE)

## ??????? ????????

In [ ]:
history_path = METRICS_DIR / f"{MODEL_NAME}_history.csv"
history_df = pd.read_csv(history_path)
history_df

In [ ]:
fig, axes = plt.subplots(1, 3 if "learning_rate" in history_df.columns else 2, figsize=(18, 5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Train loss")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Validation loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_df["epoch"], history_df["train_accuracy"], label="Train accuracy")
axes[1].plot(history_df["epoch"], history_df["val_accuracy"], label="Validation accuracy")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

if "learning_rate" in history_df.columns:
    axes[2].plot(history_df["epoch"], history_df["learning_rate"], label="Learning rate")
    axes[2].set_title("Learning rate")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()
    axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## ?????? ??????

In [ ]:
from evaluate import evaluate

evaluate(model_name=MODEL_NAME)

In [ ]:
summary_path = METRICS_DIR / f"{MODEL_NAME}_summary.json"
training_summary_path = METRICS_DIR / f"{MODEL_NAME}_training_summary.json"

with open(summary_path, "r", encoding="utf-8") as file:
    evaluation_summary = json.load(file)

print("Evaluation summary:")
display(evaluation_summary)

if training_summary_path.exists():
    with open(training_summary_path, "r", encoding="utf-8") as file:
        training_summary = json.load(file)
    print("Training summary:")
    display(training_summary)

## Confusion Matrix

In [ ]:
from IPython.display import Image, display

confusion_matrix_path = FIGURES_DIR / f"{MODEL_NAME}_confusion_matrix.png"
if confusion_matrix_path.exists():
    display(Image(filename=str(confusion_matrix_path)))
else:
    print("Confusion matrix not found:", confusion_matrix_path)